# 📈 Stock Breakout Scanner - YouTuber's Original Logic

**Original Strategy:** Price breakout confirmation using Moving Average Alignment + CAR (Cumulative Average Return)

**What This Does:**
- Downloads 2 years of stock data from Yahoo Finance
- Calculates Moving Averages (30, 50, 200 day)
- Checks if price is above all three DMAs (trend confirmation)
- Calculates CAR to ensure trend is getting stronger
- Ranks stocks by 200 DMA distance
- Exports results to Excel

## Step 1: Import Required Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import warnings
import logging
from datetime import datetime

# Suppress Yahoo Finance warnings and unnecessary logs for cleaner output
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

## Step 2: Define the Main Scanner Function

In [ ]:
def advanced_stock_scanner(ticker_list):
    """
    Scans a list of stocks for breakout signals using DMA alignment and CAR confirmation.
    
    Parameters:
    -----------
    ticker_list : list
        List of stock tickers (NSE format with .NS suffix)
    
    Returns:
    --------
    pd.DataFrame : Results dataframe sorted by 200 DMA distance
    """
    
    results = []
    today_date = datetime.now().strftime("%d-%m-%Y")
    
    print(f"🔍 Scanning {len(ticker_list)} stocks... Please wait.\n")
    
    for ticker in ticker_list:
        try:
            # Download 2 years of daily price data
            data = yf.download(ticker, period="2y", interval="1d", progress=False)
            
            # Skip stocks with insufficient data
            if data.empty or len(data) < 200:
                continue
            
            # Extract closing prices
            close_prices = data['Close'].squeeze()
            
            # Calculate Moving Averages
            dma_30 = close_prices.rolling(window=30).mean().iloc[-1]
            dma_50 = close_prices.rolling(window=50).mean().iloc[-1]
            dma_200 = close_prices.rolling(window=200).mean().iloc[-1]
            
            # Get today's closing price (Current Market Price)
            cmp = close_prices.iloc[-1]
            
            # Calculate distance from 200 DMA (how far above the trend line)
            dist_200_dma = ((cmp - dma_200) / dma_200) * 100
            
            # Find the highest high in the last year (252 trading days)
            last_1y_data = data.tail(252)
            high_date = last_1y_data['High'].squeeze().idxmax()
            
            # Extract closing prices from high date onwards
            car_data = close_prices.loc[high_date:]
            
            # Skip if insufficient CAR data
            if len(car_data) < 10:
                continue
            
            # Calculate Cumulative Average Return
            # This expands from the high date, calculating average for each day
            car_values = car_data.expanding().mean()
            
            # Get last 10 days of CAR values
            last_10_car = car_values.tail(10)
            
            # Check if CAR is monotonically increasing (trend getting stronger)
            if last_10_car.is_monotonic_increasing:
                car_status = 'Positive'
            else:
                car_status = 'Negative'
            
            # Core Filter: All three conditions must be met
            # Condition 1: Price > 30 DMA
            # Condition 2: Price > 50 DMA
            # Condition 3: Price > 200 DMA
            # Condition 4: CAR is positive (trend confirmed)
            if not ((cmp > dma_30) and (cmp > dma_50) and (cmp > dma_200) and (car_status == 'Positive')):
                continue
            
            # If we reach here, stock has passed all filters
            action = '🟢 Positive Breakout'
            
            # Store the results
            results.append({
                'Date': today_date,
                'Stock': ticker.replace('.NS', ''),
                'CMP': round(cmp, 2),
                '30 DMA': round(dma_30, 2),
                '50 DMA': round(dma_50, 2),
                '200 DMA': round(dma_200, 2),
                '200 DMA Dist %': round(dist_200_dma, 2),
                'CAR Status': car_status,
                'Action': action
            })
        
        except Exception as e:
            # Skip stocks that fail to download or process
            pass
    
    # Create DataFrame and sort by distance from 200 DMA (ascending)
    if results:
        df_positive = pd.DataFrame(results)
        df_positive = df_positive.sort_values(by='200 DMA Dist %', ascending=True)
        return df_positive
    else:
        return pd.DataFrame()

## Step 3: Define the Stock List (210 NSE Stocks)

In [ ]:
# Comprehensive list of major NSE stocks to scan
my_stocks = [
    '360ONE.NS', 'ABB.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'ADANIENSOL.NS',
    'ADANIENT.NS', 'ADANIGREEN.NS', 'ADANIPORTS.NS', 'ADANIPOWER.NS', 'ABCAPITAL.NS',
    'ALKEM.NS', 'AMBER.NS', 'AMBUJACEM.NS', 'ANGELONE.NS', 'APOLLOHOSP.NS',
    'ASHOKLEY.NS', 'ASIANPAINT.NS', 'ASTRAL.NS', 'AUROPHARMA.NS', 'DMART.NS',
    'AXISBANK.NS', 'BSE.NS', 'BAJAJ-AUTO.NS', 'BAJFINANCE.NS', 'BAJAJFINSV.NS',
    'BAJAJHLDNG.NS', 'BANDHANBNK.NS', 'BANKBARODA.NS', 'BANKINDIA.NS', 'BDL.NS',
    'BEL.NS', 'BHARATFORG.NS', 'BHEL.NS', 'BPCL.NS', 'BHARTIARTL.NS',
    'BIOCON.NS', 'BLUESTARCO.NS', 'BOSCHLTD.NS', 'BRITANNIA.NS', 'CGPOWER.NS',
    'CANBK.NS', 'CDSL.NS', 'CHOLAFIN.NS', 'CIPLA.NS', 'COALINDIA.NS',
    'COCHINSHIP.NS', 'COFORGE.NS', 'COLPAL.NS', 'CAMS.NS', 'CONCOR.NS',
    'CROMPTON.NS', 'CUMMINSIND.NS', 'DLF.NS', 'DABUR.NS', 'DALBHARAT.NS',
    'DELHIVERY.NS', 'DIVISLAB.NS', 'DIXON.NS', 'DRREDDY.NS', 'ETERNAL.NS',
    'EICHERMOT.NS', 'EXIDEIND.NS', 'FORCEMOT.NS', 'NYKAA.NS', 'FORTIS.NS',
    'GAIL.NS', 'GVTD.NS', 'GMRAIRPORT.NS', 'GLENMARK.NS', 'GODFRYPHLP.NS',
    'GODREJCP.NS', 'GODREJPROP.NS', 'GRASIM.NS', 'HCLTECH.NS', 'HDFCAMC.NS',
    'HDFCBANK.NS', 'HDFCLIFE.NS', 'HAVELLS.NS', 'HEROMOTOCO.NS', 'HINDALCO.NS',
    'HAL.NS', 'HINDPETRO.NS', 'HINDUNILVR.NS', 'HINDZINC.NS', 'POWERINDIA.NS',
    'HYUNDAI.NS', 'ICICIBANK.NS', 'ICICIGI.NS', 'ICICIPRULI.NS', 'IDFCFIRSTB.NS',
    'ITC.NS', 'INDIANB.NS', 'IEX.NS', 'IOC.NS', 'IRFC.NS', 'IREDA.NS',
    'INDUSTOWER.NS', 'INDUSINDBK.NS', 'NAUKRI.NS', 'INFY.NS', 'INOXWIND.NS',
    'INDIGO.NS', 'JINDALSTEL.NS', 'JSWENERGY.NS', 'JSWSTEEL.NS', 'JIOFIN.NS',
    'JUBLFOOD.NS', 'KEI.NS', 'KPITTECH.NS', 'KALYANIJIL.NS', 'KAYNES.NS',
    'KFINTECH.NS', 'KOTAKBANK.NS', 'LTF.NS', 'LICHSGFIN.NS', 'LTM.NS',
    'LT.NS', 'LAURUSLABS.NS', 'LICI.NS', 'LODHA.NS', 'LUPIN.NS',
    'MM.NS', 'MANAPPURAM.NS', 'MANKIND.NS', 'MARICO.NS', 'MARUTI.NS',
    'MFSL.NS', 'MAXHEALTH.NS', 'MAZDOCK.NS', 'MOTILALOFS.NS', 'MPHASIS.NS',
    'MCX.NS', 'MUTHOOTFIN.NS', 'NBCC.NS', 'NHPC.NS', 'NMDC.NS',
    'NTPC.NS', 'NATIONALUM.NS', 'NESTLEIND.NS', 'NAMINDIA.NS', 'NUVAMA.NS',
    'OBEROIRLTY.NS', 'ONGC.NS', 'OIL.NS', 'PAYTM.NS', 'OFSS.NS',
    'POLICYBZR.NS', 'PGEL.NS', 'PIIND.NS', 'PNBHOUSING.NS', 'PAGEIND.NS',
    'PATANJALI.NS', 'PERSISTENT.NS', 'PETRONET.NS', 'PIDILITIND.NS', 'POLYCAB.NS',
    'PFC.NS', 'POWERGRID.NS', 'PREMIERENE.NS', 'PRESTIGE.NS', 'PNB.NS',
    'RBLBANK.NS', 'RECLTD.NS', 'RADICO.NS', 'RVNL.NS', 'RELIANCE.NS',
    'SBICARD.NS', 'SBILIFE.NS', 'SHREECEM.NS', 'SRF.NS', 'MOTHERSON.NS',
    'SHRIRAMFIN.NS', 'SIEMENS.NS', 'SOLARINDS.NS', 'SONACOMS.NS', 'SBIN.NS',
    'SAIL.NS', 'SUNPHARMA.NS', 'SUPREMEIND.NS', 'SUZLON.NS', 'SWIGGY.NS',
    'TATACONSUM.NS', 'TVSMOTOR.NS', 'TCS.NS', 'TATAELXSI.NS', 'TMPV.NS',
    'TATAPOWER.NS', 'TATASTEEL.NS', 'TECHM.NS', 'FEDERALBNK.NS', 'INDHOTEL.NS',
    'PHOENIXLTD.NS', 'TITAN.NS', 'TORNTPHARM.NS', 'TRENT.NS', 'TIINDIA.NS',
    'UNOMINDA.NS', 'UPL.NS', 'ULTRACEMCO.NS', 'UNIONBANK.NS', 'UNITDSPR.NS',
    'VBL.NS', 'VEDL.NS', 'VMM.NS', 'IDEA.NS', 'VOLTAS.NS',
    'WAAREEENER.NS', 'WIPRO.NS', 'YESBANK.NS', 'ZYDUSLIFE.NS'
]

print(f"Total stocks to scan: {len(my_stocks)}")

## Step 4: Run the Scanner

In [ ]:
# Execute the scanner function
positive_breakout_data = advanced_stock_scanner(my_stocks)

## Step 5: Display Results

In [ ]:
# Display results with formatting
print("\n" + "="*100)
print("🟢 FINAL LIST: POSITIVE BREAKOUT STOCKS (YouTuber's Original Strategy)")
print("="*100 + "\n")

if positive_breakout_data.empty:
    print("❌ No stocks matched the criteria today.\n")
else:
    print(positive_breakout_data.to_string(index=False))
    print("\n" + "="*100)

## Step 6: Export to Excel

In [ ]:
# Save results to Excel file
if not positive_breakout_data.empty:
    filename = "YouTuber_Breakout_Scanner_Results.xlsx"
    positive_breakout_data.to_excel(filename, index=False)
    print(f"✅ Results saved to '{filename}'")
    
    # Display summary
    print(f"\nTotal Breakout Stocks Found: {len(positive_breakout_data)}")
    print(f"Average Distance from 200 DMA: {positive_breakout_data['200 DMA Dist %'].mean():.2f}%")
else:
    print("No data to export.")

## How to Interpret Results

### Column Explanations:

| Column | Meaning |
|--------|----------|
| **Date** | Scan date (DD-MM-YYYY format) |
| **Stock** | Stock ticker (without .NS suffix) |
| **CMP** | Current Market Price (today's close) |
| **30 DMA** | 30-day Moving Average |
| **50 DMA** | 50-day Moving Average |
| **200 DMA** | 200-day Moving Average (main trend line) |
| **200 DMA Dist %** | How far (%) the stock is above its 200-day average |
| **CAR Status** | Cumulative Average Return status (Positive/Negative) |
| **Action** | Trading signal (🟢 Positive Breakout) |

### What Each Filter Means:

1. **Price > 30 DMA**: Stock price above short-term average (momentum)
2. **Price > 50 DMA**: Stock price above medium-term average (consistency)
3. **Price > 200 DMA**: Stock price above long-term trend (major uptrend)
4. **CAR = Positive**: Trend is getting STRONGER (not just maintaining)

### Trading Interpretation:

Stocks that pass all four filters are in a **strong uptrend** with:
- ✅ Short-term momentum
- ✅ Medium-term consistency  
- ✅ Long-term trend alignment
- ✅ Strengthening pattern

These are **high-probability breakout candidates** for trend-following traders.